# WordPress on OCI — OpenTofu

Two WordPress VMs and a load balancer in a **public subnet**. MySQL HeatWave PaaS in a **private DB subnet** with no internet route. SSH to VMs is restricted to admin IP only.

## OCI CLI

Install if not already present:
```bash
bash -c "$(curl -L https://raw.githubusercontent.com/oracle/oci-cli/master/scripts/install/install.sh)"
```

In [ ]:
!which oci

In [ ]:
!oci session refresh --profile harleycalvert

## Install OpenTofu

[Set Up OCI Terraform](https://docs.oracle.com/en-us/iaas/developer-tutorials/tutorials/tf-provider/01-summary.htm)

In [ ]:
!snap install --classic opentofu

In [ ]:
!tofu -v

## Generate SSH key — Run in Terminal
```
ssh-keygen -t rsa -b 4096 -f ~/.ssh/wp_vm_key
```

In [ ]:
!cat ~/.ssh/wp_vm_key.pub

## TLS Certificate

Use Let's Encrypt for a trusted cert.


## Structure

```
wordpress/
  provider.tf      main.tf          variables.tf
  outputs.tf       terraform.tfvars
  modules/
    compartment/   main.tf  variables.tf  outputs.tf
    vcn/           main.tf  security_lists.tf  variables.tf  outputs.tf
    nsg/           main.tf  variables.tf  outputs.tf
    compute/       main.tf  variables.tf  outputs.tf
    lb/            main.tf  variables.tf  outputs.tf
    mysql/         main.tf  variables.tf  outputs.tf
```

**Network layout:**
```
Internet ──443/80──► Load Balancer (public subnet 10.0.0.0/24)
                           │ :80
                    wp-01  wp-02  (public subnet 10.0.0.0/24)
                           │ :3306
                    MySQL HeatWave  (db subnet 10.0.1.0/24, no internet)

Admin ──:22──► wp-01 / wp-02  (direct SSH, restricted to admin IP)
```

In [ ]:
!mkdir -p modules/compartment modules/vcn modules/nsg modules/compute modules/lb modules/mysql

In [ ]:
%%writefile ./provider.tf

provider "oci" {
  config_file_profile = "harleycalvert"
}


# Root

In [ ]:
%%writefile ./main.tf

module "compartment" {
  source           = "./modules/compartment"
  tenancy_id       = var.tenancy_id
  compartment_name = var.compartment_name
}

module "vcn" {
  source             = "./modules/vcn"
  compartment_id     = module.compartment.id
  vcn_cidr           = var.vcn_cidr
  public_subnet_cidr = var.public_subnet_cidr
  db_subnet_cidr     = var.db_subnet_cidr
  admin_ip           = var.admin_ip
}

module "nsg" {
  source         = "./modules/nsg"
  compartment_id = module.compartment.id
  vcn_id         = module.vcn.vcn_id
  admin_ip       = var.admin_ip
  db_subnet_cidr = var.db_subnet_cidr
}

module "compute" {
  source   = "./modules/compute"
  for_each = toset(["wp-01", "wp-02"])

  tenancy_id          = var.tenancy_id
  compartment_id      = module.compartment.id
  subnet_id           = module.vcn.public_subnet_id
  nsg_ids             = [module.nsg.app_nsg_id]
  assign_public_ip    = true
  ssh_public_key_path = var.ssh_public_key_path
  display_name        = each.key
  hostname            = each.key
  shape               = var.shape
  image_id            = var.image_id
}

locals {
  backend_ips = {
    for name, vm in module.compute :
    name => vm.private_ip
  }
}

module "lb" {
  source         = "./modules/lb"
  compartment_id = module.compartment.id
  subnet_id      = module.vcn.public_subnet_id
  nsg_ids        = [module.nsg.lb_nsg_id]
  backend_ips    = local.backend_ips

  ca_certificate_path     = var.ca_certificate_path
  public_certificate_path = var.public_certificate_path
  private_key_path        = var.private_key_path
}

module "mysql" {
  source         = "./modules/mysql"
  tenancy_id     = var.tenancy_id
  compartment_id = module.compartment.id
  subnet_id      = module.vcn.db_subnet_id

  admin_username = var.mysql_admin_username
  admin_password = var.mysql_admin_password
  shape_name     = var.mysql_shape
}


In [ ]:
%%writefile ./variables.tf

variable "tenancy_id" {
  type        = string
  description = "OCID of the tenancy"
}

variable "compartment_name" {
  type        = string
  description = "Name of the OCI compartment to create"
}

variable "vcn_cidr" {
  type        = string
  description = "CIDR block for the VCN"
}

variable "public_subnet_cidr" {
  type        = string
  description = "CIDR for the public subnet (load balancer + VMs)"
}

variable "db_subnet_cidr" {
  type        = string
  description = "CIDR for the DB subnet (MySQL HeatWave — no internet route)"
}

variable "shape" {
  type        = string
  description = "Compute instance shape"
}

variable "image_id" {
  type        = string
  description = "OCID of the compute image"
}

variable "admin_ip" {
  type        = string
  description = "Admin public IP (CIDR) — SSH access is restricted to this address"
}

variable "ssh_public_key_path" {
  type        = string
  description = "Path to SSH public key"
}

variable "mysql_admin_username" {
  type        = string
  description = "MySQL HeatWave admin username"
}

variable "mysql_admin_password" {
  type        = string
  sensitive   = true
  description = "MySQL HeatWave admin password: 8-32 chars, upper+lower+digit+special"
}

variable "mysql_shape" {
  type        = string
  default     = "MySQL.Free"
  description = "MySQL HeatWave shape (MySQL.Free for Always Free tier)"
}

variable "ca_certificate_path" {
  type        = string
  description = "Path to CA certificate PEM"
}

variable "public_certificate_path" {
  type        = string
  description = "Path to public certificate PEM"
}

variable "private_key_path" {
  type        = string
  description = "Path to private key PEM"
}


In [ ]:
%%writefile ./terraform.tfvars

# this file is not submitted to Git - add to .gitignore

tenancy_id           = "<your-tenancy_id>"
compartment_name     = "wordpress"
vcn_cidr             = "10.0.0.0/16"
public_subnet_cidr   = "10.0.0.0/24"
db_subnet_cidr       = "10.0.1.0/24"
# https://docs.oracle.com/en-us/iaas/images/
# Use Canonical-Ubuntu-22.04 for your region
shape                = "VM.Standard.E2.1.Micro"
image_id             = "<ocid1.image.oc1.ap-melbourne-1.xxxxx>"
admin_ip             = "<your-public-ip>/32"  # <YOUR_PUBLIC_IP>/32
ssh_public_key_path  = "~/.ssh/wp_vm_key.pub"

mysql_admin_username = "wpadmin"
mysql_admin_password = "<top-secret-password>"  # replace before apply — 8-32 chars, upper+lower+digit+special
mysql_shape          = "MySQL.Free"

ca_certificate_path     = "~/.tls/chain.pem"
public_certificate_path = "~/.tls/fullchain.pem"
private_key_path        = "~/.tls/privkey.pem"


In [ ]:
%%writefile ./outputs.tf

output "lb_public_ip" {
  value = module.lb.public_ip
}

output "vm_public_ips" {
  value = {
    for name, vm in module.compute :
    name => vm.public_ip
  }
}

output "mysql_ip" {
  value = module.mysql.ip_address
}

output "mysql_port" {
  value = module.mysql.port
}


# Compartment

`time_sleep` gives OCI IAM 30 s to propagate the new compartment before any VCN or compute resources attempt to use it.

In [ ]:
%%writefile ./modules/compartment/main.tf

resource "oci_identity_compartment" "compartment" {
  compartment_id = var.tenancy_id
  name           = var.compartment_name
  description    = "Managed by OpenTofu"
  enable_delete  = true
}

resource "time_sleep" "wait" {
  depends_on      = [oci_identity_compartment.compartment]
  create_duration = "30s"
}


In [ ]:
%%writefile ./modules/compartment/variables.tf

variable "tenancy_id" {
  type = string
}

variable "compartment_name" {
  type = string
}


In [ ]:
%%writefile ./modules/compartment/outputs.tf

output "id" {
  value      = oci_identity_compartment.compartment.id
  depends_on = [time_sleep.wait]
}


# VCN

**Public subnet** (`10.0.0.0/24`) — Load Balancer and WordPress VMs. Routes to Internet Gateway. VMs are assigned public IPs.

**DB subnet** (`10.0.1.0/24`) — MySQL HeatWave only. `prohibit_public_ip_on_vnic = true`. Route table has **no rules** — zero internet path in or out. Intra-VCN routing (VM → MySQL) works automatically without route rules.

**Security lists** enforce subnet-boundary rules: HTTPS/HTTP/SSH on the public SL; MySQL 3306 from the public subnet on the DB SL. NSGs enforce per-resource rules on top.

In [ ]:
%%writefile ./modules/vcn/main.tf

resource "oci_core_vcn" "vcn" {
  compartment_id = var.compartment_id
  cidr_block     = var.vcn_cidr
  display_name   = "vcn"
  dns_label      = "vcn"
}

resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "igw"
  enabled        = true
}

resource "oci_core_route_table" "public_rt" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "public-rt"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}

resource "oci_core_route_table" "db_rt" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "db-rt"
  # No route rules — DB subnet has no internet path
}

resource "oci_core_subnet" "public" {
  compartment_id  = var.compartment_id
  vcn_id          = oci_core_vcn.vcn.id
  cidr_block      = var.public_subnet_cidr
  display_name    = "public-subnet"
  route_table_id  = oci_core_route_table.public_rt.id
  dhcp_options_id = data.oci_core_dhcp_options.default.options[0].id
  dns_label       = "public"

  security_list_ids = [oci_core_security_list.public_sl.id]
}

resource "oci_core_subnet" "db" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  cidr_block                 = var.db_subnet_cidr
  display_name               = "db-subnet"
  route_table_id             = oci_core_route_table.db_rt.id
  dhcp_options_id            = data.oci_core_dhcp_options.default.options[0].id
  dns_label                  = "db"
  prohibit_public_ip_on_vnic = true

  security_list_ids = [oci_core_security_list.db_sl.id]
}

data "oci_core_dhcp_options" "default" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "Default DHCP Options for vcn"
}


In [ ]:
%%writefile ./modules/vcn/security_lists.tf

resource "oci_core_security_list" "public_sl" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "public-sl"

  # HTTPS from internet (load balancer)
  ingress_security_rules {
    protocol = "6"
    source   = "0.0.0.0/0"

    tcp_options {
      min = 443
      max = 443
    }
  }

  # HTTP from internet (load balancer)
  ingress_security_rules {
    protocol = "6"
    source   = "0.0.0.0/0"

    tcp_options {
      min = 80
      max = 80
    }
  }

  # SSH from admin IP only
  ingress_security_rules {
    protocol = "6"
    source   = var.admin_ip

    tcp_options {
      min = 22
      max = 22
    }
  }

  ingress_security_rules {
    protocol = "1"
    source   = "0.0.0.0/0"

    icmp_options {
      type = 3
      code = 4
    }
  }

  egress_security_rules {
    protocol    = "all"
    destination = "0.0.0.0/0"
  }
}

resource "oci_core_security_list" "db_sl" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "db-sl"

  # MySQL from app (public) subnet only — subnet-boundary layer on top of NSG
  ingress_security_rules {
    protocol = "6"
    source   = var.public_subnet_cidr

    tcp_options {
      min = 3306
      max = 3306
    }
  }

  ingress_security_rules {
    protocol = "1"
    source   = "0.0.0.0/0"

    icmp_options {
      type = 3
      code = 4
    }
  }

  egress_security_rules {
    protocol    = "all"
    destination = "0.0.0.0/0"
  }
}


In [ ]:
%%writefile ./modules/vcn/variables.tf

variable "compartment_id" {
  type = string
}

variable "vcn_cidr" {
  type = string
}

variable "public_subnet_cidr" {
  type = string
}

variable "db_subnet_cidr" {
  type        = string
  description = "CIDR for the DB subnet (MySQL HeatWave — no internet route)"
}

variable "admin_ip" {
  type        = string
  description = "Admin public IP (CIDR) — allowed SSH in public security list"
}


In [ ]:
%%writefile ./modules/vcn/outputs.tf

output "vcn_id" {
  value = oci_core_vcn.vcn.id
}

output "public_subnet_id" {
  value = oci_core_subnet.public.id
}

output "db_subnet_id" {
  value = oci_core_subnet.db.id
}


# NSG — Network Security Groups

Two NSGs for the public subnet resources. MySQL HeatWave is protected at the subnet boundary by `db_sl` (security list) — no NSG is attached to it, which avoids a known provisioning failure with managed PaaS services.

| NSG | Ingress | Egress |
|-----|---------|--------|
| **lb-nsg** | 80, 443 from `0.0.0.0/0` | :80 → app-nsg |
| **app-nsg** | :80 from lb-nsg · :22 from `admin_ip` | :3306 → db subnet CIDR · :80/:443 internet |

SSH restricted to `admin_ip` at both NSG and security list — two independent enforcement points. MySQL access from VMs is allowed by both the app-nsg egress rule (CIDR-scoped to `db_subnet_cidr`) and the `db_sl` ingress rule.

In [ ]:
%%writefile ./modules/nsg/main.tf

# --- Load Balancer NSG ---
resource "oci_core_network_security_group" "lb" {
  compartment_id = var.compartment_id
  vcn_id         = var.vcn_id
  display_name   = "lb-nsg"
}

# --- App (WordPress VM) NSG ---
resource "oci_core_network_security_group" "app" {
  compartment_id = var.compartment_id
  vcn_id         = var.vcn_id
  display_name   = "app-nsg"
}

# LB: HTTPS ingress from internet
resource "oci_core_network_security_group_security_rule" "lb_ingress_https" {
  network_security_group_id = oci_core_network_security_group.lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source                    = "0.0.0.0/0"

  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}

# LB: HTTP ingress from internet
resource "oci_core_network_security_group_security_rule" "lb_ingress_http" {
  network_security_group_id = oci_core_network_security_group.lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source                    = "0.0.0.0/0"

  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# LB: egress HTTP to App NSG (backends)
resource "oci_core_network_security_group_security_rule" "lb_egress_to_app" {
  network_security_group_id = oci_core_network_security_group.lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.app.id

  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# App: ingress HTTP from LB NSG
resource "oci_core_network_security_group_security_rule" "app_ingress_from_lb" {
  network_security_group_id = oci_core_network_security_group.app.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.lb.id

  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# App: SSH ingress from admin IP only
resource "oci_core_network_security_group_security_rule" "app_ingress_ssh" {
  network_security_group_id = oci_core_network_security_group.app.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source                    = var.admin_ip

  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}

# App: egress MySQL to DB subnet (security list handles subnet-boundary)
resource "oci_core_network_security_group_security_rule" "app_egress_to_db" {
  network_security_group_id = oci_core_network_security_group.app.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination               = var.db_subnet_cidr

  tcp_options {
    destination_port_range {
      min = 3306
      max = 3306
    }
  }
}

# App: egress HTTPS to internet (WordPress plugin/apt updates)
resource "oci_core_network_security_group_security_rule" "app_egress_https" {
  network_security_group_id = oci_core_network_security_group.app.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination               = "0.0.0.0/0"

  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}

# App: egress HTTP to internet (apt)
resource "oci_core_network_security_group_security_rule" "app_egress_http" {
  network_security_group_id = oci_core_network_security_group.app.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination               = "0.0.0.0/0"

  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}


In [ ]:
%%writefile ./modules/nsg/variables.tf

variable "compartment_id" {
  type = string
}

variable "vcn_id" {
  type = string
}

variable "admin_ip" {
  type        = string
  description = "Admin public IP (CIDR) — SSH to VMs is restricted to this address"
}

variable "db_subnet_cidr" {
  type        = string
  description = "DB subnet CIDR — app-nsg MySQL egress is scoped to this CIDR"
}


In [ ]:
%%writefile ./modules/nsg/outputs.tf

output "lb_nsg_id" {
  value = oci_core_network_security_group.lb.id
}

output "app_nsg_id" {
  value = oci_core_network_security_group.app.id
}


# Compute

VMs in the **public subnet** with `assign_public_ip = true`. No public IPs are assigned via the NSG — only the LB (port 80) and admin IP (port 22) have inbound paths. `replace(hostname, "-", "")` removes hyphens — OCI `hostname_label` requires alphanumeric only.

In [ ]:
%%writefile ./modules/compute/main.tf

data "oci_identity_availability_domains" "ads" {
  compartment_id = var.tenancy_id
}

resource "oci_core_instance" "vm" {
  availability_domain = data.oci_identity_availability_domains.ads.availability_domains[var.ad_index].name
  compartment_id      = var.compartment_id
  display_name        = var.display_name
  shape               = var.shape

  source_details {
    source_type = "image"
    source_id   = var.image_id
  }

  create_vnic_details {
    subnet_id        = var.subnet_id
    nsg_ids          = var.nsg_ids
    assign_public_ip = var.assign_public_ip
    hostname_label   = replace(var.hostname, "-", "")
  }

  metadata = {
    ssh_authorized_keys = file(pathexpand(var.ssh_public_key_path))
  }

  preserve_boot_volume = false
}


In [ ]:
%%writefile ./modules/compute/variables.tf

variable "tenancy_id" {
  type = string
}

variable "compartment_id" {
  type = string
}

variable "subnet_id" {
  type = string
}

variable "nsg_ids" {
  type = list(string)
}

variable "ssh_public_key_path" {
  type = string
}

variable "display_name" {
  type = string
}

variable "hostname" {
  type = string
}

variable "shape" {
  type = string
}

variable "image_id" {
  type = string
}

variable "assign_public_ip" {
  type    = bool
  default = true
}

variable "ad_index" {
  type    = number
  default = 0
}


In [ ]:
%%writefile ./modules/compute/outputs.tf

output "instance_id" {
  value = oci_core_instance.vm.id
}

output "public_ip" {
  value = oci_core_instance.vm.public_ip
}

output "private_ip" {
  value = oci_core_instance.vm.private_ip
}


# Load Balancer

Flexible shape (10/10 Mbps, Always Free) in the **public subnet**. HTTPS listener with TLS termination — backends receive plain HTTP on port 80.

In [ ]:
%%writefile ./modules/lb/main.tf

resource "oci_load_balancer_load_balancer" "lb" {
  compartment_id             = var.compartment_id
  display_name               = var.display_name
  shape                      = "flexible"
  subnet_ids                 = [var.subnet_id]
  network_security_group_ids = var.nsg_ids

  shape_details {
    minimum_bandwidth_in_mbps = 10
    maximum_bandwidth_in_mbps = 10
  }
}

resource "oci_load_balancer_backend_set" "bs" {
  name             = var.backend_set_name
  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  policy           = "ROUND_ROBIN"

  health_checker {
    protocol = "HTTP"
    url_path = "/"
    port     = var.backend_port
  }
}

resource "oci_load_balancer_backend" "backend" {
  for_each = var.backend_ips

  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  backendset_name  = oci_load_balancer_backend_set.bs.name
  ip_address       = each.value
  port             = var.backend_port
  weight           = 1
}

resource "oci_load_balancer_certificate" "cert" {
  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  certificate_name = var.certificate_name

  ca_certificate     = file(pathexpand(var.ca_certificate_path))
  public_certificate = file(pathexpand(var.public_certificate_path))
  private_key        = file(pathexpand(var.private_key_path))

  lifecycle {
    create_before_destroy = true
  }
}

# resource "oci_load_balancer_listener" "http_listener" {
#   load_balancer_id         = oci_load_balancer_load_balancer.lb.id
#   name                     = "http-listener"
#   default_backend_set_name = oci_load_balancer_backend_set.bs.name
#   port                     = 80
#   protocol                 = "HTTP"
# }

resource "oci_load_balancer_listener" "https_listener" {
  load_balancer_id         = oci_load_balancer_load_balancer.lb.id
  name                     = "https-listener"
  default_backend_set_name = oci_load_balancer_backend_set.bs.name
  port                     = 443
  protocol                 = "HTTP" # OCI uses HTTP even for TLS termination

  ssl_configuration {
    certificate_name        = oci_load_balancer_certificate.cert.certificate_name
    verify_peer_certificate = false
    cipher_suite_name       = "oci-default-ssl-cipher-suite-v1"
    protocols               = ["TLSv1.2", "TLSv1.3"]
  }
}


In [ ]:
%%writefile ./modules/lb/variables.tf

variable "compartment_id" {
  type = string
}

variable "subnet_id" {
  type = string
}

variable "nsg_ids" {
  type = list(string)
}

variable "display_name" {
  type    = string
  default = "lb"
}

variable "backend_set_name" {
  type    = string
  default = "wp-backend-set"
}

variable "backend_port" {
  type    = number
  default = 80
}

variable "backend_ips" {
  type        = map(string)
  description = "Map of VM name to private IP for LB backends"
}

variable "certificate_name" {
  type    = string
  default = "wp-cert"
}

variable "ca_certificate_path" {
  type        = string
  description = "Path to CA certificate PEM"
}

variable "public_certificate_path" {
  type        = string
  description = "Path to public certificate PEM"
}

variable "private_key_path" {
  type        = string
  description = "Path to private key PEM"
}


In [ ]:
%%writefile ./modules/lb/outputs.tf

output "public_ip" {
  value = oci_load_balancer_load_balancer.lb.ip_address_details[0].ip_address
}


In [ ]:
%%writefile ./modules/mysql/main.tf

data "oci_identity_availability_domains" "ads" {
  compartment_id = var.tenancy_id
}

resource "oci_mysql_mysql_db_system" "mysql" {
  compartment_id      = var.compartment_id
  availability_domain = data.oci_identity_availability_domains.ads.availability_domains[0].name

  display_name = "wordpress-mysql"
  shape_name   = var.shape_name

  admin_username = var.admin_username
  admin_password = var.admin_password

  subnet_id = var.subnet_id
  # nsg_ids not set — the DB security list (db_sl) enforces subnet-boundary
  # access control; NSGs on managed PaaS services can cause FAILED provisioning

  data_storage_size_in_gb = 50
  is_highly_available     = false

  deletion_policy {
    automatic_backup_retention = "RETAIN"
    final_backup               = "SKIP_FINAL_BACKUP"
    is_delete_protected        = false
  }

  maintenance {
    window_start_time = "sun 02:00"
  }
}


In [ ]:
%%writefile ./modules/mysql/variables.tf

variable "tenancy_id" {
  type = string
}

variable "compartment_id" {
  type = string
}

variable "subnet_id" {
  type = string
}

variable "admin_username" {
  type = string
}

variable "admin_password" {
  type      = string
  sensitive = true
}

variable "shape_name" {
  type        = string
  default     = "MySQL.Free"
  description = "MySQL HeatWave shape — MySQL.Free for Always Free tier"
}


In [ ]:
%%writefile ./modules/mysql/outputs.tf

output "id" {
  value = oci_mysql_mysql_db_system.mysql.id
}

output "ip_address" {
  value = oci_mysql_mysql_db_system.mysql.ip_address
}

output "port" {
  value = oci_mysql_mysql_db_system.mysql.port
}


In [ ]:
!tree -al

In [ ]:
!tofu init

In [ ]:
!tofu validate

In [ ]:
!tofu plan

In [ ]:
!tofu apply -auto-approve

In [ ]:
!tofu output

# MySQL HeatWave

`MySQL.Free` shape: 1 OCPU · 8 GB RAM · 50 GB storage. Always Free tier.
`is_highly_available = false` — single-node (HA requires 3 nodes and is not free tier).
Lives in the **DB subnet** — no internet route. Only reachable from `app-nsg` on port 3306, enforced at both the `db_sl` security list and `db-nsg`.

**After apply — set up the WordPress database:**
```bash
# SSH directly to a VM (public IP from tofu output):
ssh -i ~/.ssh/wp_vm_key ubuntu@<vm_public_ip>

# Install MySQL client and connect:
sudo apt install -y mysql-client
mysql -h <mysql_ip> -u wpadmin -p

# In MySQL:
CREATE DATABASE wordpress CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
CREATE USER 'wpuser'@'%' IDENTIFIED BY 'StrongPass123!';
GRANT ALL PRIVILEGES ON wordpress.* TO 'wpuser'@'%';
FLUSH PRIVILEGES;
EXIT;
```

## Install WordPress (run on each VM)

```bash
ssh -i ~/.ssh/wp_vm_key ubuntu@<vm_public_ip>

sudo apt update && sudo apt install -y \
  apache2 php php-mysql libapache2-mod-php \
  php-curl php-gd php-mbstring php-xml php-xmlrpc

cd /tmp && wget https://wordpress.org/latest.tar.gz
sudo tar -xzf latest.tar.gz -C /var/www/html --strip-components=1
sudo chown -R www-data:www-data /var/www/html
sudo a2enmod rewrite && sudo systemctl restart apache2

sudo cp /var/www/html/wp-config-sample.php /var/www/html/wp-config.php
sudo sed -i "s/wordpress/wordpress/" /var/www/html/wp-config.php
sudo sed -i "s/wpuser/wpuser/"         /var/www/html/wp-config.php
sudo sed -i "s/password_here/StrongPass123!/" /var/www/html/wp-config.php
sudo sed -i "s/localhost/<mysql_ip>/"         /var/www/html/wp-config.php

iptables -I INPUT 1 -m state --state NEW -p tcp --dport 80 -j ACCEPT
iptables -I INPUT 1 -m state --state NEW -p tcp --dport 443 -j ACCEPT
netfilter-persistent save

sudo vim /var/www/html/wp-config.php

define('WP_HOME', 'https://yourdomain.com');
define('WP_SITEURL', 'https://yourdomain.com');

if (isset($_SERVER['HTTP_X_FORWARDED_PROTO']) && $_SERVER['HTTP_X_FORWARDED_PROTO'] === 'https') {
    $_SERVER['HTTPS'] = 'on';
}
```
Open `https://<lb_public_ip>` to complete the WordPress setup wizard.

`https://<your-domain>/wp-config.php`

`https://<your-domain>/wp-login.php`

In [ ]:
!tofu destroy -auto-approve